# Titanic 27: random_strength + subsample + rsm randomized 5-Fold CV

`titanic_15.ipynb`에서 실제 `submission/titanic_result_15.csv`를 만든 모델을 기준으로 한 독립 하이퍼파라미터 실험이다.
기준 피처는 `GenderClass`, `GenderIsChild`, `ClassIsChild`, `AgeBand`이며, 전처리/결측치/인코딩/holdout split/평가/제출 형식은 Titanic 15와 동일하게 유지한다.

선택에는 train 데이터의 5-Fold CV ROC-AUC만 사용한다. test는 최종 예측에만 사용한다.

In [1]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
from common.interaction_experiments import FeaturePreprocessor
from common.feature_experiments import baseline_parameters

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

train = pd.read_csv('csv/train.csv')
test = pd.read_csv('csv/test.csv')
submission_template = pd.read_csv('csv/submission.csv')

target_col = 'survived'
id_col = 'passengerid'
BASE_FEATURES = ['GenderClass', 'GenderIsChild', 'ClassIsChild', 'AgeBand']
X = train.drop(columns=[target_col, id_col])
y = train[target_col].astype('int8')
X_test_raw = test.drop(columns=[id_col])

assert X.columns.equals(X_test_raw.columns)
assert train[id_col].is_unique and test[id_col].is_unique
assert set(train[id_col]).isdisjoint(test[id_col])

## Random State Audit

Python/NumPy 전역 seed를 Kernel Restart 후 처음부터 고정한다. 모든 데이터 분할과 탐색 샘플링은 명시적인 `random_state=42`, CatBoost는 Titanic 15의 `random_state=42`(CatBoost 내부 `random_seed=42`와 같은 alias)를 사용한다.

In [2]:
RANDOM_AUDIT = pd.DataFrame([
    ('Python random', 42, 'random.seed(42)'),
    ('NumPy', 42, 'np.random.seed(42)'),
    ('train_test_split', 42, 'explicit random_state'),
    ('StratifiedKFold', 42, 'shuffle=True, explicit random_state'),
    ('CatBoost', 42, 'baseline random_state alias -> random_seed'),
    ('ParameterSampler', 42, 'explicit random_state'),
], columns=['Component', 'Random State', 'Code basis'])
display(RANDOM_AUDIT)
assert RANDOM_AUDIT.loc[RANDOM_AUDIT['Random State'].ne('N/A'), 'Random State'].eq(42).all()
print('모든 사용 stochastic component의 seed가 코드에서 42로 고정되었습니다.')

,Component,Random State,Code basis
0,Python random,42,random.seed(42)
1,NumPy,42,np.random.seed(42)
2,train_test_split,42,explicit random_state
3,StratifiedKFold,42,"shuffle=True, explicit random_state"
4,CatBoost,42,baseline random_state alias -> random_seed
5,ParameterSampler,42,explicit random_state


모든 사용 stochastic component의 seed가 코드에서 42로 고정되었습니다.


## 고정된 데이터 분할과 누수 방지 구조

Titanic 15의 기본 25% stratified holdout을 같은 seed로 재현한다. CV에서는 매 fold마다 새 전처리기를 만들고 Train Fold에만 `fit_transform`, Validation Fold에는 `transform`만 적용한다.

In [3]:
train_part, valid_part = train_test_split(
    train, test_size=0.25, stratify=train[target_col], random_state=SEED
)
X_tr_raw = train_part.drop(columns=[target_col, id_col])
y_tr = train_part[target_col].astype('int8')
X_valid_raw = valid_part.drop(columns=[target_col, id_col])
y_valid = valid_part[target_col].astype('int8')
assert len(train_part) == 687 and len(valid_part) == 229

BASE_PARAMS = baseline_parameters()
assert BASE_PARAMS['random_state'] == SEED
print('Titanic 15 명시 파라미터:', BASE_PARAMS)
print('고정 피처:', BASE_FEATURES)

Titanic 15 명시 파라미터: {'verbose': 0, 'random_state': 42, 'cat_features': [], 'allow_writing_files': False}
고정 피처: ['GenderClass', 'GenderIsChild', 'ClassIsChild', 'AgeBand']


In [4]:
def prepare_fold(X_train_raw, X_valid_raw):
    prep = FeaturePreprocessor(BASE_FEATURES)
    X_train_model = prep.fit_transform(X_train_raw)
    X_valid_model = prep.transform(X_valid_raw)
    assert X_train_model.columns.equals(X_valid_model.columns)
    assert np.isfinite(X_train_model.to_numpy()).all()
    assert np.isfinite(X_valid_model.to_numpy()).all()
    return prep, X_train_model, X_valid_model


def fit_model(X_train_model, y_train, overrides=None, **fit_kwargs):
    params = BASE_PARAMS | (overrides or {})
    model = CatBoostClassifier(**params)
    model.fit(X_train_model, y_train, **fit_kwargs)
    return model


def auc_scores(model, X_train_model, y_train, X_valid_model, y_valid):
    positive_index = list(model.classes_).index(1)
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train_model)[:, positive_index])
    valid_auc = roc_auc_score(y_valid, model.predict_proba(X_valid_model)[:, positive_index])
    return train_auc, valid_auc, train_auc - valid_auc


def baseline_cv():
    rows = []
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
        _, X_fold_train, X_fold_valid = prepare_fold(X.iloc[tr_idx], X.iloc[va_idx])
        model = fit_model(X_fold_train, y.iloc[tr_idx])
        train_auc, valid_auc, gap = auc_scores(
            model, X_fold_train, y.iloc[tr_idx], X_fold_valid, y.iloc[va_idx]
        )
        rows.append({'fold': fold, 'train_auc': train_auc,
                     'validation_auc': valid_auc, 'gap': gap})
    folds = pd.DataFrame(rows)
    summary = {
        'cv_mean': folds['validation_auc'].mean(),
        'cv_std': folds['validation_auc'].std(ddof=1),
        'train_auc': folds['train_auc'].mean(),
        'gap': folds['gap'].mean(),
    }
    return folds, summary


baseline_folds, baseline_cv_summary = baseline_cv()
display(baseline_folds)
print('Titanic 15 actual-submission baseline CV:', baseline_cv_summary)

,fold,train_auc,validation_auc,gap
0,1,0.957912,0.906140,0.051772
1,2,0.958143,0.924358,0.033785
2,3,0.963202,0.886728,0.076475
3,4,0.956995,0.905861,0.051135
4,5,0.957728,0.914505,0.043222


Titanic 15 actual-submission baseline CV: {'cv_mean': np.float64(0.9075184337655735), 'cv_std': np.float64(0.013868055749294898), 'train_auc': np.float64(0.9587961681724849), 'gap': np.float64(0.05127773440691141)}


## 고정된 36개 조합 Randomized Search

전체 60개 중 `ParameterSampler(random_state=42)`로 36개 조합을 고정 추출한다. `random_strength`, `subsample`, `rsm`만 바꾼다.

In [5]:
from sklearn.model_selection import ParameterSampler

search_space = {
    'random_strength': [0.2, 0.5, 1, 2, 4],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'rsm': [0.7, 0.85, 1.0],
}
sampled_params = list(ParameterSampler(search_space, n_iter=36, random_state=SEED))
assert len(sampled_params) == 36
assert len({tuple(sorted(p.items())) for p in sampled_params}) == 36

candidate_rows = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for params in sampled_params:
    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), 1):
        _, X_fold_train, X_fold_valid = prepare_fold(X.iloc[tr_idx], X.iloc[va_idx])
        model = fit_model(X_fold_train, y.iloc[tr_idx], params)
        train_auc, valid_auc, gap = auc_scores(
            model, X_fold_train, y.iloc[tr_idx], X_fold_valid, y.iloc[va_idx]
        )
        candidate_rows.append({**params, 'fold': fold, 'train_auc': train_auc,
                               'validation_auc': valid_auc, 'gap': gap})
    print('완료:', params)

cv_folds = pd.DataFrame(candidate_rows)
cv_summary = (cv_folds.groupby(['random_strength', 'subsample', 'rsm'], as_index=False)
              .agg(cv_mean=('validation_auc', 'mean'),
                   cv_std=('validation_auc', 'std'),
                   train_auc=('train_auc', 'mean'),
                   gap=('gap', 'mean'))
              .sort_values(['cv_mean', 'cv_std'], ascending=[False, True], ignore_index=True))
display(cv_folds)
display(cv_summary)
best = cv_summary.iloc[0]
BEST_TUNED = {'random_strength': float(best['random_strength']),
              'subsample': float(best['subsample']), 'rsm': float(best['rsm'])}
print('Selected Best Parameter:', BEST_TUNED)

완료: {'subsample': 0.7, 'rsm': 0.7, 'random_strength': 0.2}


완료: {'subsample': 0.8, 'rsm': 0.85, 'random_strength': 0.2}


완료: {'subsample': 0.7, 'rsm': 0.7, 'random_strength': 2}


완료: {'subsample': 0.8, 'rsm': 1.0, 'random_strength': 2}


완료: {'subsample': 0.8, 'rsm': 0.7, 'random_strength': 0.5}


...

완료: {'subsample': 1.0, 'rsm': 1.0, 'random_strength': 0.2}


완료: {'subsample': 0.7, 'rsm': 1.0, 'random_strength': 1}


,subsample,rsm,random_strength,fold,train_auc,validation_auc,gap
0,0.7,0.7,0.2,1,0.964209,0.905890,0.058319
1,0.7,0.7,0.2,2,0.965518,0.923722,0.041796
2,0.7,0.7,0.2,3,0.970771,0.883931,0.086840
3,0.7,0.7,0.2,4,0.965716,0.905225,0.060491
4,0.7,0.7,0.2,5,0.967921,0.915205,0.052716
...,...,...,...,...,...,...,...
175,0.7,1.0,1.0,1,0.957555,0.908145,0.049409
176,0.7,1.0,1.0,2,0.958413,0.924612,0.033800
177,0.7,1.0,1.0,3,0.964675,0.887999,0.076676
178,0.7,1.0,1.0,4,0.959165,0.902428,0.056736


,random_strength,subsample,rsm,cv_mean,cv_std,train_auc,gap
0,0.2,0.8,1.00,0.909254,0.015288,0.967199,0.057945
1,1.0,0.7,0.70,0.908888,0.012778,0.958011,0.049123
2,0.5,1.0,0.70,0.908770,0.016024,0.963058,0.054287
3,2.0,0.7,0.85,0.908713,0.012145,0.954337,0.045624
4,0.2,0.7,1.00,0.908343,0.014845,0.968019,0.059676
5,4.0,0.7,0.85,0.908324,0.011613,0.949914,0.041591
6,0.2,0.8,0.85,0.908252,0.015711,0.966947,0.058695
7,4.0,0.9,1.00,0.908249,0.011782,0.949236,0.040987
8,0.2,0.7,0.85,0.908201,0.013713,0.967569,0.059369
9,4.0,0.7,0.70,0.908180,0.012220,0.949505,0.041325


Selected Best Parameter: {'random_strength': 0.2, 'subsample': 0.8, 'rsm': 1.0}


In [6]:
_, X_holdout_train, X_holdout_valid = prepare_fold(X_tr_raw, X_valid_raw)
baseline_holdout_model = fit_model(X_holdout_train, y_tr)
baseline_holdout = auc_scores(baseline_holdout_model, X_holdout_train, y_tr,
                              X_holdout_valid, y_valid)
best_holdout_model = fit_model(X_holdout_train, y_tr, BEST_TUNED)
best_holdout = auc_scores(best_holdout_model, X_holdout_train, y_tr,
                          X_holdout_valid, y_valid)

comparison = pd.DataFrame([
    {'Model': 'Baseline Titanic 15', 'Parameter': str(BASE_PARAMS),
     'CV Mean': baseline_cv_summary['cv_mean'], 'CV Std': baseline_cv_summary['cv_std'],
     'Validation AUC': baseline_holdout[1], 'Gap': baseline_holdout[2]},
    {'Model': 'Titanic 27 Best', 'Parameter': str(BEST_TUNED),
     'CV Mean': best['cv_mean'], 'CV Std': best['cv_std'],
     'Validation AUC': best_holdout[1], 'Gap': best_holdout[2]},
])
display(comparison)
print('Final Parameter:', BEST_TUNED)
FINAL_PARAMS = BASE_PARAMS | BEST_TUNED

,Model,Parameter,CV Mean,CV Std,Validation AUC,Gap
0,Baseline Titanic 15,"{'verbose': 0, 'random_state': 42, 'cat_featur...",0.907518,0.013868,0.900024,0.062311
1,Titanic 27 Best,"{'random_strength': 0.2, 'subsample': 0.8, 'rs...",0.909254,0.015288,0.895633,0.077763


Final Parameter: {'random_strength': 0.2, 'subsample': 0.8, 'rsm': 1.0}


## 최종 모델과 제출 파일

선택된 파라미터로 전체 train에 전처리를 fit하고 test에는 transform만 적용한다. test는 CV, 파라미터 선택, Early Stopping에 사용하지 않는다.

In [7]:
final_prep = FeaturePreprocessor(BASE_FEATURES)
X_full_model = final_prep.fit_transform(X)
X_test_model = final_prep.transform(X_test_raw)
assert X_full_model.columns.equals(X_test_model.columns)
final_model = CatBoostClassifier(**FINAL_PARAMS)
final_model.fit(X_full_model, y)
positive_index = list(final_model.classes_).index(1)
predictions = final_model.predict_proba(X_test_model)[:, positive_index]

# Titanic 15와 같은 template/id mapping 방식으로 생성한다.
assert submission_template.columns.tolist() == [id_col, target_col]
assert len(submission_template) == len(test)
assert submission_template[id_col].is_unique and submission_template[id_col].notna().all()
assert set(submission_template[id_col]) == set(test[id_col])
result = submission_template.copy(deep=True)
by_id = pd.Series(predictions, index=test[id_col].to_numpy())
result[target_col] = result[id_col].map(by_id)

# 저장 직전 검증: 행 수, ID/순서, 확률 dtype, NaN, 범위.
assert len(result) == len(test) == len(submission_template)
pd.testing.assert_series_equal(result[id_col], submission_template[id_col], check_names=True)
assert pd.api.types.is_float_dtype(result[target_col])
assert result[target_col].notna().all()
assert np.isfinite(result[target_col].to_numpy()).all()
assert result[target_col].between(0.0, 1.0).all()

output_path = Path('titanic_27_result.csv')
result.to_csv(output_path, index=False)
saved = pd.read_csv(output_path)
assert saved.shape == submission_template.shape == (len(test), 2)
assert saved.columns.equals(submission_template.columns)
pd.testing.assert_frame_equal(saved.drop(columns=target_col), submission_template.drop(columns=target_col))
assert pd.api.types.is_float_dtype(saved[target_col])
assert saved[target_col].notna().all() and saved[target_col].between(0.0, 1.0).all()
np.testing.assert_allclose(saved[target_col], by_id.loc[submission_template[id_col]], rtol=1e-12, atol=1e-15)
print('Submission 검증 통과:', output_path, saved.shape)

Submission 검증 통과: titanic_27_result.csv (393, 2)
